In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Dataset Description

## Overview

The dataset pertains to forest cover types in the Roosevelt National Forest of northern Colorado. Each observation represents a 30m x 30m patch of land, and the goal is to predict the forest cover type from various features.

## Target Variable

- **Cover_Type**: Integer classification of forest cover type. The possible values are:
  - 1: Spruce/Fir
  - 2: Lodgepole Pine
  - 3: Ponderosa Pine
  - 4: Cottonwood/Willow
  - 5: Aspen
  - 6: Douglas-fir
  - 7: Krummholz

## Features

1. **Elevation**: Elevation in meters.
2. **Aspect**: Aspect in degrees azimuth.
3. **Slope**: Slope in degrees.
4. **Horizontal_Distance_To_Hydrology**: Horizontal distance to the nearest surface water features.
5. **Vertical_Distance_To_Hydrology**: Vertical distance to the nearest surface water features.
6. **Horizontal_Distance_To_Roadways**: Horizontal distance to the nearest roadway.
7. **Hillshade_9am**: Hillshade index at 9 am, summer solstice (0 to 255 index).
8. **Hillshade_Noon**: Hillshade index at noon, summer solstice (0 to 255 index).
9. **Hillshade_3pm**: Hillshade index at 3 pm, summer solstice (0 to 255 index).
10. **Horizontal_Distance_To_Fire_Points**: Horizontal distance to the nearest wildfire ignition points.
11. **Wilderness_Area**: Four binary columns indicating wilderness area designation:
    - 1: Rawah Wilderness Area
    - 2: Neota Wilderness Area
    - 3: Comanche Peak Wilderness Area
    - 4: Cache la Poudre Wilderness Area
12. **Soil_Type**: Forty binary columns indicating soil type designation:
    - Various soil types with specific characteristics (see full list in the dataset description).

## Training and Test Sets

- **Training Set**: Contains 15,120 observations with both features and the target variable (`Cover_Type`).
- **Test Set**: Contains 565,892 observations with features only. Predictions for `Cover_Type` are required for this set.


# Forest Cover Type Prediction
<img src='https://turkiyeraporu.com/wp-content/uploads/2022/12/orman_tw.jpg'>

<br>
<br>

# Input  Data

In [ ]:
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

In [ ]:
train.head()

In [ ]:
train.info()


In [ ]:
test.isnull().sum()

In [ ]:
train.describe()

# EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(50, 50))

corr = train.corr(numeric_only=True)

sns.heatmap(corr, annot=True, cmap='coolwarm')

plt.title('Heatmap')
plt.show()

In [ ]:
features_to_plot = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology']
plt.figure(figsize=(14, 10))
for i, feature in enumerate(features_to_plot, 1):
    plt.subplot(2, 2, i)
    sns.histplot(train[feature], kde=True, bins=30)
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Cover_Type']

# Create the pairplot
pairplot = sns.pairplot(train[features_to_plot], hue='Cover_Type', palette='viridis', diag_kind='kde')

# Customize the plot
pairplot.fig.suptitle('Pairplot of Selected Features', y=1.02)
plt.show()

# Modeling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Özellikleri ölçeklendirme
features = train.drop(columns=['Cover_Type', 'Id'])
scaler = StandardScaler()
features = scaler.fit_transform(features)


## Test the other Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

b = BernoulliNB()
l = LogisticRegression()
d = DecisionTreeClassifier()
r = RandomForestClassifier()
gb= GradientBoostingClassifier()
kn= KNeighborsClassifier()
ab= AdaBoostClassifier()


def algo_test(x, y):
    modeller=[ b, l, d, r, gb, kn, ab]
    isimler=["BernoulliNB", "LogisticRegression", "DecisionTreeClassifier", 
             "RandomForestClassifier", "GradientBoostingClassifier", "KNeighborsClassifier",
             "AdaBoostClassifier"]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.3, random_state = 42)
    
    accuracy = []
    precision = []
    recall = []
    f1 = []
    mdl=[]

    print("Veriler hazır modeller deneniyor\n")
    for model in modeller:
        print(model, " model fiting!..\n")
        model=model.fit(x_train,y_train)
        tahmin=model.predict(x_test)
        mdl.append(model)
        accuracy.append(accuracy_score(y_test, tahmin))
        precision.append(precision_score(y_test, tahmin, average="micro"))
        recall.append(recall_score(y_test, tahmin, average="micro"))
        f1.append(f1_score(y_test, tahmin, average="micro"))
        #print(confusion_matrix(y_test, tahmin))

    print("Training completed.\n")
    
    metrics=pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1", "Model"], index=isimler)
    metrics["Accuracy"] = accuracy
    metrics["Precision"] = precision  
    metrics["Recall"] = recall
    metrics["F1"] = f1
    metrics["Model"]=mdl

    metrics.sort_values("F1", ascending=False, inplace=True)

    print("\n****Best Model**** : ", metrics.iloc[0].name)
    model=metrics.iloc[0,-1]
    tahmin=model.predict(np.array(x_test) if model==kn else x_test)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, tahmin))
    print("\nclassification Report:")
    print(classification_report(y_test, tahmin))
    print("\nDiğer Modeller:")
    
    return metrics.drop("Model", axis=1)

In [ ]:
from sklearn.model_selection import train_test_split as tts
 
x = features
y = train['Cover_Type']

x_train, x_val, y_train, y_val = tts(x, y, test_size=0.2, random_state=42)


In [ ]:
x_train.shape,x_val.shape

In [ ]:
import warnings 
warnings.filterwarnings('ignore')
algo_test(x,y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

In [ ]:
rf.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score as ac

y_pred = rf.predict(x_val)

acc = ac(y_val, y_pred)

print('Accuracy : ',acc)

In [ ]:
y_pred

# Deep Learning

In [ ]:
# Rearrange labels to start from 0
y_train = y_train - 1
y_val = y_val - 1

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

model = Sequential([
    Dense(128, activation='relu', input_shape=(x_train.shape[1],)),  # İlk katman
    Dense(64, activation='relu'),  
    Dense(7, activation='softmax')  # 7 class
])

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history = model.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_val, y_val))

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### Confusion Matrix

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

y_pred = model.predict(x_val)
y_pred_classes = np.argmax(y_pred, axis=1)  

cm = confusion_matrix(y_val, y_pred_classes)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Class 1', 'Class 2', 'Class 3', 'Class 4', 'Class 5', 'Class 6', 'Class 7'],
            yticklabels=['Class 1', 'Class 2', 'Class 3', 'Class 4', 'Class 5', 'Class 6', 'Class 7'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


### Classification Rapor

In [ ]:
from sklearn.metrics import classification_report

# Assuming y_val (true labels) and y_pred_classes (predicted labels) are defined
report = classification_report(y_val, y_pred_classes, target_names=[
    'Spruce/Fir',
    'Lodgepole Pine',
    'Ponderosa Pine',
    'Cottonwood/Willow',
    'Aspen',
    'Douglas-fir',
    'Krummholz'
])

print("Classification Report:")
print(report)

## Main Mission

In [ ]:
# Test
test_feature = scaler.transform(test.drop(columns=['Id']))
test_pred = rf.predict(test_feature)

In [ ]:
submission = pd.DataFrame({'Id': test['Id'], 'Cover_Type': test_pred})
submission.to_csv('submission.csv', index=False)

### The End